In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load original dataset
df = pd.read_csv("C:/Users/sivanand/Desktop/Start-/dataset/online_retail_II.csv", encoding='utf-8', low_memory=False)
print(f"Original shape: {df.shape}")

# Remove cancellations
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Remove bad values
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]
df = df.dropna(subset=['Description'])

# Remove extreme quantities
df = df[df['Quantity'] <= 10000]

# Safe copy
df = df.copy()

# Convert date — we need this for daily forecasting!
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Create Revenue
df['Revenue'] = df['Quantity'] * df['Price']

# Extract time features
df['Year']       = df['InvoiceDate'].dt.year
df['Month']      = df['InvoiceDate'].dt.month
df['Day']        = df['InvoiceDate'].dt.day
df['DayOfWeek']  = df['InvoiceDate'].dt.dayofweek
df['IsWeekend']  = df['DayOfWeek'].isin([5, 6]).astype(int)
df['Quarter']    = df['InvoiceDate'].dt.quarter
df['WeekOfYear'] = df['InvoiceDate'].dt.isocalendar().week.astype(int)

print(f"After cleaning: {df.shape}")

Original shape: (1067371, 8)
After cleaning: (1041663, 16)


In [3]:
# Group tiny countries
top_countries = df['Country'].value_counts()
top_countries = top_countries[top_countries >= 500].index
df['Country_Grouped'] = df['Country'].apply(
    lambda x: x if x in top_countries else 'Other'
)

# Encode
le_country = LabelEncoder()
df['Country_Encoded'] = le_country.fit_transform(df['Country_Grouped'])
print(f"Countries encoded: {df['Country_Encoded'].nunique()}")

Countries encoded: 22


In [4]:
# Clean description text
df['Description'] = df['Description'].str.upper().str.strip()

# Keep most common description per StockCode
df['Description'] = df.groupby('StockCode')['Description']\
                       .transform(lambda x: x.mode()[0])

# Encode
le_desc = LabelEncoder()
df['Description_Encoded'] = le_desc.fit_transform(df['Description'])
print(f"Products encoded: {df['Description_Encoded'].nunique()}")

Products encoded: 4716


In [5]:
# Group by date
daily = df.groupby(df['InvoiceDate'].dt.date).agg(
    Total_Revenue    = ('Revenue', 'sum'),
    Total_Orders     = ('Description_Encoded', 'count'),
    Total_Quantity   = ('Quantity', 'sum'),
    Unique_Products  = ('Description_Encoded', 'nunique'),
    Unique_Countries = ('Country_Encoded', 'nunique')
).reset_index()

# Fix date column
daily = daily.rename(columns={'InvoiceDate': 'Date'})
daily['Date'] = pd.to_datetime(daily['Date'])

# Add time features
daily['Year']       = daily['Date'].dt.year
daily['Month']      = daily['Date'].dt.month
daily['Day']        = daily['Date'].dt.day
daily['DayOfWeek']  = daily['Date'].dt.dayofweek
daily['IsWeekend']  = daily['DayOfWeek'].isin([5, 6]).astype(int)
daily['Quarter']    = daily['Date'].dt.quarter
daily['WeekOfYear'] = daily['Date'].dt.isocalendar().week.astype(int)

# Sort chronologically
daily = daily.sort_values('Date').reset_index(drop=True)

print(f"Total days: {len(daily)}")
print(f"Date range: {daily['Date'].min()} → {daily['Date'].max()}")
print(daily.head(10))

Total days: 604
Date range: 2009-12-01 00:00:00 → 2011-12-09 00:00:00
        Date  Total_Revenue  Total_Orders  Total_Quantity  Unique_Products  \
0 2009-12-01       54513.50          3105           26204             1314   
1 2009-12-02       63352.51          3250           31896             1386   
2 2009-12-03       74037.91          2946           49243             1315   
3 2009-12-04       40732.92          2528           21325             1250   
4 2009-12-05        9803.05           400            5119              284   
5 2009-12-06       24613.64          1911           11623              962   
6 2009-12-07       45083.35          2824           18058             1334   
7 2009-12-08       49517.23          2189           23159             1126   
8 2009-12-09       40616.09          2433           17934             1295   
9 2009-12-10       44442.11          2403           23718             1209   

   Unique_Countries  Year  Month  Day  DayOfWeek  IsWeekend  Quarter  \

In [6]:
# Lag features — past days as inputs
daily['lag_1']     = daily['Total_Revenue'].shift(1)
daily['lag_2']     = daily['Total_Revenue'].shift(2)
daily['lag_3']     = daily['Total_Revenue'].shift(3)
daily['lag_7']     = daily['Total_Revenue'].shift(7)   # same day last week
daily['rolling_7'] = daily['Total_Revenue'].shift(1).rolling(7).mean()
daily['rolling_3'] = daily['Total_Revenue'].shift(1).rolling(3).mean()

# Drop NaN rows from lag
daily = daily.dropna().reset_index(drop=True)

print(f"Days available for modelling: {len(daily)}")
print(f"Columns: {daily.columns.tolist()}")
print(daily.head())

Days available for modelling: 597
Columns: ['Date', 'Total_Revenue', 'Total_Orders', 'Total_Quantity', 'Unique_Products', 'Unique_Countries', 'Year', 'Month', 'Day', 'DayOfWeek', 'IsWeekend', 'Quarter', 'WeekOfYear', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'rolling_7', 'rolling_3']
        Date  Total_Revenue  Total_Orders  Total_Quantity  Unique_Products  \
0 2009-12-08       49517.23          2189           23159             1126   
1 2009-12-09       40616.09          2433           17934             1295   
2 2009-12-10       44442.11          2403           23718             1209   
3 2009-12-11       39659.60          1439           21394              836   
4 2009-12-13       22313.09          1767           12611              973   

   Unique_Countries  Year  Month  Day  DayOfWeek  IsWeekend  Quarter  \
0                 6  2009     12    8          1          0        4   
1                 3  2009     12    9          2          0        4   
2                 4  2009     12   1

In [9]:
daily

,Date,Total_Revenue,Total_Orders,Total_Quantity,Unique_Products,Unique_Countries,Year,Month,Day,DayOfWeek,IsWeekend,Quarter,WeekOfYear,lag_1,lag_2,lag_3,lag_7,rolling_7,rolling_3
0,2009-12-08,49517.23,2189,23159,1126,6,2009,12,8,1,0,4,50,45083.35,24613.64,9803.05,54513.50,44590.982857,26500.013333
1,2009-12-09,40616.09,2433,17934,1295,3,2009,12,9,2,0,4,50,49517.23,45083.35,24613.64,63352.51,43877.230000,39738.073333
2,2009-12-10,44442.11,2403,23718,1209,4,2009,12,10,3,0,4,50,40616.09,49517.23,45083.35,74037.91,40629.170000,45072.223333
3,2009-12-11,39659.60,1439,21394,836,1,2009,12,11,4,0,4,50,44442.11,40616.09,49517.23,40732.92,36401.198571,44858.476667
4,2009-12-13,22313.09,1767,12611,973,5,2009,12,13,6,1,4,50,39659.60,44442.11,40616.09,9803.05,36247.867143,41572.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
592,2011-12-05,88741.96,5297,44578,1748,10,2011,12,5,0,0,4,49,24621.43,57664.07,52197.25,20611.08,49283.130000,44827.583333
593,2011-12-06,56713.21,3262,30468,1420,7,2011,12,6,1,0,4,49,88741.96,24621.43,57664.07,57165.19,59016.112857,57009.153333
594,2011-12-07,75439.16,2390,41880,1108,8,2011,12,7,2,0,4,49,56713.21,88741.96,24621.43,72595.93,58951.544286,56692.200000
595,2011-12-08,82495.00,4867,35085,1652,6,2011,12,8,3,0,4,49,75439.16,56713.21,88741.96,60126.96,59357.720000,73631.443333


In [10]:
# Select only the columns you want
final_df = daily[['Year', 'Month', 'Day', 'DayOfWeek', 'IsWeekend',
                  'Total_Orders', 'lag_1', 'lag_2', 'lag_7',
                  'rolling_7', 'rolling_3', 'Total_Revenue']]

# Save to CSV
final_df.to_csv('mlreset1.csv', index=False)

print(f"✅ Saved successfully!")

✅ Saved successfully!


In [12]:
final_df.to_csv(r'C:/Users/sivanand/Desktop/internship/online_retail_III.csv', index=False)